# Feature Engineering — NLP Features
Reads processed_posts from reddit_warehouse.db, computes NLP features (VADER sentiment, NRC emotions, readability) keyed on post_id for the XGBoost model.

- vader_compound: sentiment, -1 to +1
- nrc_*: 8 emotion scores (joy, trust, fear, surprise, sadness, disgust, anger, anticipation)
- empty text -> NaN, text with no emotion words -> 0.0

In [2]:
import duckdb


# notebook lives in notebooks/zoe/, db is at project root, so go up two levels
con = duckdb.connect("../../reddit_warehouse.db")

print(con.execute("SHOW TABLES").df())
posts = con.execute("SELECT post_id, title, body FROM processed_posts").df()
print(posts.shape)
posts.head()

              name
0    post_features
1      post_labels
2  processed_posts
3        raw_posts
(5744, 3)


,post_id,title,body
0,1ckm56h,What is the best sunscreen for oily skin? (wit...,
1,19fjsnj,What is one makeup tip you have found to be a ...,
2,1cmgd8q,"I accepted that I'll likely never find a gf, h...",I (m 33) never had a gf. Much less ever been o...
3,1cha0s7,I have $10k from a 401k I need to move.,Former employer went bankrupt and folded. I gu...
4,1d9o2s2,My CPU cores are higher than it supposed to?,I was looking for an upgrade to Ryzen 5 5600 f...


In [3]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import numpy as np

analyzer = SentimentIntensityAnalyzer()

def get_vader_compound(title, body):
    # combine title + body; this is your swappable text scope
    text = f"{title} {body}".strip()
    if not text:
        return np.nan          # no text at all -> missing, not 0
    return analyzer.polarity_scores(text)["compound"]

posts["vader_compound"] = posts.apply(
    lambda row: get_vader_compound(row["title"], row["body"]), axis=1
)

posts[["title", "body", "vader_compound"]].head(10)

,title,body,vader_compound
0,What is the best sunscreen for oily skin? (wit...,,0.6369
1,What is one makeup tip you have found to be a ...,,0.0000
2,"I accepted that I'll likely never find a gf, h...",I (m 33) never had a gf. Much less ever been o...,0.6838
3,I have $10k from a 401k I need to move.,Former employer went bankrupt and folded. I gu...,-0.5574
4,My CPU cores are higher than it supposed to?,I was looking for an upgrade to Ryzen 5 5600 f...,0.5632
5,How do you cope with being physically ugly?,One thing I've struggled with is being objecti...,0.9513
6,[Homemade] Pizza!,,0.0000
7,Practice cakes good enough to start selling?,I made these three cakes with vanilla and smbc...,-0.6601
8,"Who will China send? Shi, Tian, or Li?","Assuming gigachad Lihuanhua does well, who wil...",0.6083
9,[Homemade] Chicken Tenders and Hot Honey Mustard,,0.1531


In [8]:
import urllib.request, os
import numpy as np
from collections import defaultdict

LEX_PATH = "nrc_lexicon.txt"
if not os.path.exists(LEX_PATH):
    url = "https://raw.githubusercontent.com/dinbav/LeXmo/master/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt"
    urllib.request.urlretrieve(url, LEX_PATH)

EMOTIONS = ["joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"]
word_emotions = defaultdict(set)

with open(LEX_PATH) as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) != 3:          # skip headers, blank lines, anything malformed
            continue
        word, emotion, flag = parts
        if flag == "1" and emotion in EMOTIONS:
            word_emotions[word].add(emotion)

print("lexicon loaded, words:", len(word_emotions))

lexicon loaded, words: 4463


In [9]:
import re

def get_emotions(title, body):
    text = f"{title} {body}".strip().lower()
    if not text:
        # no text at all -> missing, not 0
        return {f"nrc_{e}": np.nan for e in EMOTIONS}

    words = re.findall(r"[a-z']+", text)   # simple word tokenizer
    if not words:
        return {f"nrc_{e}": np.nan for e in EMOTIONS}

    # count emotion hits across all words
    counts = {e: 0 for e in EMOTIONS}
    for w in words:
        for emotion in word_emotions.get(w, ()):
            counts[emotion] += 1

    # normalize by total words so long posts don't score higher just for length
    n = len(words)
    return {f"nrc_{e}": counts[e] / n for e in EMOTIONS}

emotion_cols = posts.apply(
    lambda row: get_emotions(row["title"], row["body"]),
    axis=1, result_type="expand"
)
posts = posts.join(emotion_cols)

posts[["title", "nrc_anger", "nrc_joy", "nrc_fear", "nrc_sadness"]].head(10)

,title,nrc_anger,nrc_joy,nrc_fear,nrc_sadness
0,What is the best sunscreen for oily skin? (wit...,0.000000,0.000000,0.000000,0.000000
1,What is one makeup tip you have found to be a ...,0.000000,0.071429,0.000000,0.000000
2,"I accepted that I'll likely never find a gf, h...",0.010417,0.031250,0.031250,0.020833
3,I have $10k from a 401k I need to move.,0.018182,0.018182,0.018182,0.018182
4,My CPU cores are higher than it supposed to?,0.000000,0.000000,0.000000,0.000000
5,How do you cope with being physically ugly?,0.008439,0.029536,0.012658,0.016878
6,[Homemade] Pizza!,0.000000,0.000000,0.000000,0.000000
7,Practice cakes good enough to start selling?,0.009434,0.009434,0.009434,0.009434
8,"Who will China send? Shi, Tian, or Li?",0.020408,0.000000,0.020408,0.020408
9,[Homemade] Chicken Tenders and Hot Honey Mustard,0.142857,0.000000,0.142857,0.000000


In [10]:
posts.to_parquet("zoe_features_wip.parquet")
print("saved:", posts.shape)
print(posts.columns.tolist())

saved: (5744, 12)
['post_id', 'title', 'body', 'vader_compound', 'nrc_joy', 'nrc_trust', 'nrc_fear', 'nrc_surprise', 'nrc_sadness', 'nrc_disgust', 'nrc_anger', 'nrc_anticipation']
